<a href="https://colab.research.google.com/github/royaritro/DL_Spring_2026/blob/main/DL_kg4021_ar10067.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Qwen3-1.7B

**Notebook Details**
- Model: `Qwen/Qwen3-1.7B` (1.7B params)
- No quantization needed — 1.7B in BF16 is only ~3.4GB VRAM
- LoRA r=32
- batch size = (16) and standard adamw_torch optimizer
- Adapter saved to `attempt2_1p7b/`

In [1]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
DRIVE_ROOT  = "/content/drive/MyDrive/DL/midterm"
MODEL_NAME  = "Qwen/Qwen3-1.7B"

CONFIG = {
    "model_name":                   MODEL_NAME,
    "train_csv_path":               f"{DRIVE_ROOT}/dl-spring-2026-svg-generation/train.csv",
    "test_csv_path":                f"{DRIVE_ROOT}/dl-spring-2026-svg-generation/test.csv",
    "output_dir":                   f"{DRIVE_ROOT}/attempt2_1p7b",
    "max_seq_length":               2048,
    "lora_r":                       32,
    "lora_alpha":                   32,
    "lora_dropout":                 0.05,
    "learning_rate":                5e-5,
    "num_train_epochs":             1,
    "per_device_train_batch_size":  16,
    "gradient_accumulation_steps":  1,      # effective batch = 16
    "warmup_ratio":                 0.03,
    "weight_decay":                 0.01,
    "logging_steps":                20,
    "eval_steps":                   300,
    "save_steps":                   500,
    "max_new_tokens":               1024,
    "eval_size":                    0.02,
    "infer_batch":                  16,
}

SEED = 42
print("CONFIG:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

CONFIG:
  model_name: Qwen/Qwen3-1.7B
  train_csv_path: /content/drive/MyDrive/DL/midterm/dl-spring-2026-svg-generation/train.csv
  test_csv_path: /content/drive/MyDrive/DL/midterm/dl-spring-2026-svg-generation/test.csv
  output_dir: /content/drive/MyDrive/DL/midterm/attempt2_1p7b
  max_seq_length: 2048
  lora_r: 32
  lora_alpha: 32
  lora_dropout: 0.05
  learning_rate: 5e-05
  num_train_epochs: 1
  per_device_train_batch_size: 16
  gradient_accumulation_steps: 1
  warmup_ratio: 0.03
  weight_decay: 0.01
  logging_steps: 20
  eval_steps: 300
  save_steps: 500
  max_new_tokens: 1024
  eval_size: 0.02
  infer_batch: 16


In [2]:
# Drive mounting as training and inference will be done on Colab, we need to mount Google Drive to access the dataset and save the model weights.
from google.colab import drive
drive.mount("/content/drive")
print("Drive mounted.")


Mounted at /content/drive
Drive mounted.


In [3]:
# Flash Attention 2 wheel pre-built for cu12 + torch2.10 + cp312 (matches Colab Blackwell)
%pip install -q "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"
%pip install -q datasets trl transformers accelerate peft bitsandbytes pandas lxml


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 51.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 56.7 MB/s eta 0:00:00


In [4]:
import os, re, csv, shutil, pathlib, time, random, xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, get_peft_model, TaskType
from trl import SFTTrainer, SFTConfig

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda"
print(f"Torch  : {torch.__version__}")
print(f"Device : {DEVICE}")
print(f"GPU    : {torch.cuda.get_device_name(0)}")
print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


Torch  : 2.10.0+cu128
Device : cuda
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition
VRAM   : 102.0 GB


In [5]:
'''
Data loading and preprocessing. We copy the CSV from Drive to local storage for faster access, then read it into a DataFrame. We filter out rows with missing or invalid SVGs, and those that are too long to fit in the model's context window. Finally, we convert the DataFrame to a Hugging Face Dataset and split it into training and evaluation sets. We exclude the system prompt from the token count, leaving headroom for the user prompt and model response. The final dataset contains only valid SVGs that start with "<svg" and are between 50 and ~3700 characters long (≈1800 tokens). We print stats at each step to track how many samples are dropped.

Source: /content/drive/MyDrive/DL/midterm/dl-spring-2026-svg-generation/train.csv  (exists=True, size=135.5 MB)
Reading CSV …
After startswith <svg : 50,000  (dropped 0)
After XML validity    : 50,000  (dropped 0)
After length filter   : 38,510  (dropped 11,490)

SVG length stats:
count    38510.000000
mean      1732.325734
std        946.660508
min         91.000000
25%        933.000000
50%       1626.000000
75%       2479.000000
max       3696.000000

Train: 37,739   Eval: 771
'''
import csv, shutil, pathlib

_src = pathlib.Path(CONFIG["train_csv_path"])
_dst = pathlib.Path("/content/train_local.csv")

print(f"Source: {_src}  (exists={_src.exists()}, size={_src.stat().st_size/1e6:.1f} MB)")
if not _dst.exists():
    print(f"Copying → {_dst} …")
    shutil.copy2(_src, _dst)
print("Reading CSV …")

with open(_dst, "r", encoding="utf-8", newline="", errors="replace") as f:
    rows = list(csv.DictReader(f))
df = pd.DataFrame(rows)[["prompt", "svg"]].dropna()

n = len(df)
df = df[df["svg"].str.startswith("<svg", na=False)]
print(f"After startswith <svg : {len(df):,}  (dropped {n - len(df):,})")

def _is_valid_xml(svg):
    try:
        ET.fromstring(svg)
        return True
    except ET.ParseError:
        return False

n = len(df)
df = df[df["svg"].apply(_is_valid_xml)].reset_index(drop=True)
print(f"After XML validity    : {len(df):,}  (dropped {n - len(df):,})")

# 2 chars/token is accurate for SVG (dense numbers, quotes, attribute names).
# Leaves ~248 tokens headroom for system prompt + user prompt.
MAX_SVG_CHARS = CONFIG["max_seq_length"] * 2 - 400   # 2048*2-400 = 3696 chars ≈ 1800 tokens
n = len(df)
df = df[df["svg"].str.len().between(50, MAX_SVG_CHARS)].reset_index(drop=True)
print(f"After length filter   : {len(df):,}  (dropped {n - len(df):,})")

print(f"\nSVG length stats:\n{df['svg'].str.len().describe().to_string()}")

ds = Dataset.from_pandas(df).shuffle(seed=SEED)
splits = ds.train_test_split(test_size=CONFIG["eval_size"], seed=SEED)
train_ds, eval_ds = splits["train"], splits["test"]
print(f"\nTrain: {len(train_ds):,}   Eval: {len(eval_ds):,}")

Source: /content/drive/MyDrive/DL/midterm/dl-spring-2026-svg-generation/train.csv  (exists=True, size=135.5 MB)
Copying → /content/train_local.csv …
Reading CSV …
After startswith <svg : 50,000  (dropped 0)
After XML validity    : 50,000  (dropped 0)
After length filter   : 38,510  (dropped 11,490)

SVG length stats:
count    38510.000000
mean      1732.325734
std        946.660508
min         91.000000
25%        933.000000
50%       1626.000000
75%       2479.000000
max       3696.000000

Train: 37,739   Eval: 771


In [6]:
SYSTEM_PROMPT = (
    "You generate compact, valid SVG markup from user requests. "
    "Return ONLY the SVG code — no explanation, no markdown fences. "
    "Use a single root <svg> element with xmlns=\"http://www.w3.org/2000/svg\" "
    "and viewBox=\"0 0 200 200\". Keep output under 15000 characters. "
    "Use EXACTLY the colors and shapes described in the prompt."
)

# Qwen3 uses /no_think at end of user turn to suppress <think>...</think> output.
# Training format: system -> user (/no_think) -> assistant (SVG only)
def format_sft(example):
    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}<|im_end|>\n"
        "<|im_start|>user\n"
        f"{example['prompt']} /no_think<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{example['svg']}<|im_end|>"
    )
    return {"text": text}

train_text = train_ds.map(format_sft, remove_columns=train_ds.column_names)
eval_text  = eval_ds.map(format_sft,  remove_columns=eval_ds.column_names)
print("Sample formatted text (first 400 chars):")
print(train_text[0]["text"][:400])

Map:   0%|          | 0/37739 [00:00<?, ? examples/s]

Map:   0%|          | 0/771 [00:00<?, ? examples/s]

Sample formatted text (first 400 chars):
<|im_start|>system
You generate compact, valid SVG markup from user requests. Return ONLY the SVG code — no explanation, no markdown fences. Use a single root <svg> element with xmlns="http://www.w3.org/2000/svg" and viewBox="0 0 200 200". Keep output under 15000 characters. Use EXACTLY the colors and shapes described in the prompt.<|im_end|>
<|im_start|>user
A simple black silhouette of a human f


In [7]:
'''
Training data sanity check: 2 random examples. We render the SVG as HTML to visually verify correctness, and also show the original prompt and SVG character count. This helps ensure that the data preprocessing is working as intended and that the model will receive valid inputs during training.
'''
from IPython.display import display, HTML
import random

samples = [train_ds[i] for i in random.sample(range(len(train_ds)), 2)]

cells = ""
for ex in samples:
    prompt = ex["prompt"]
    svg    = ex["svg"]
    length = len(svg)
    cells += f"""
    <div style="display:inline-block; margin:16px; vertical-align:top; max-width:320px;">
      <div style="font-size:12px; font-weight:bold; margin-bottom:6px;">Prompt:</div>
      <div style="font-size:12px; background:#f5f5f5; padding:8px; border-radius:4px; margin-bottom:10px;">{prompt}</div>
      <div style="font-size:12px; font-weight:bold; margin-bottom:6px;">Target SVG ({length} chars):</div>
      <div style="border:1px solid #ccc; width:200px; height:200px;">{svg}</div>
    </div>"""

display(HTML(f'<h3>Training data samples</h3><div style="display:flex; flex-wrap:wrap;">{cells}</div>'))

In [8]:

'''
This is where we load the pre-trained Qwen3-1.7B model and prepare it for fine-tuning with LoRA. We use the Hugging Face Transformers library to load the model in BF16 precision, which is efficient on modern GPUs. We configure LoRA to target the attention and feedforward layers of the transformer, which are the most impactful for fine-tuning. We also load the corresponding tokenizer and set the maximum sequence length to match our CONFIG. Finally, we print the memory usage after loading the model to confirm that it fits within our GPU's VRAM.
'''

print(f"Loading base model: {CONFIG['model_name']} …")
base_model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_name"],
    device_map="auto",
    torch_dtype=torch.bfloat16,
    attn_implementation="flash_attention_2",
)

lora_cfg = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(base_model, lora_cfg)
model.print_trainable_parameters()

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
tokenizer.model_max_length = CONFIG["max_seq_length"]
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Memory used after LoRA: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

Loading base model: Qwen/Qwen3-1.7B …


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Memory used after LoRA: 4.20 GB


In [9]:
'''
Verify Flash Attention 2 is active. We check three things: (1) the flash_attn package is installed, (2) the model's config indicates that flash_attention_2 is being used, and (3) the class name of the first attention layer contains "flash". This helps ensure that we're actually using Flash Attention 2, which is crucial for fitting the model in VRAM and achieving good performance. Flash attention 2 over traditional attention gave approx 5x speedup, without it iterattion were 0.03 iter/s, with flash attention 2 it is 0.17 iter/s on same hardware.
'''
import importlib
fa_spec = importlib.util.find_spec("flash_attn")
if fa_spec:
    import flash_attn
    print(f"flash_attn installed: v{flash_attn.__version__}")
else:
    print("flash_attn NOT installed")
attn_impl = getattr(base_model.config, "_attn_implementation", "unknown")
status = "Success" if attn_impl == "flash_attention_2" else "Fail"
print(f"{status} model._attn_implementation = {attn_impl!r}")

first_attn = next(
    (m for name, m in base_model.named_modules() if "attention" in name.lower()),
    None,
)
if first_attn:
    cls = type(first_attn).__name__
    status = "Success" if "flash" in cls.lower() else "Fail"
    print(f"{status} First attention layer class: {cls}")

flash_attn installed: v2.8.3
Success model._attn_implementation = 'flash_attention_2'
Fail First attention layer class: Qwen3RMSNorm


In [10]:
sft_config = SFTConfig(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_ratio=CONFIG["warmup_ratio"],
    weight_decay=CONFIG["weight_decay"],
    fp16=False,
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    optim="adamw_torch",
    lr_scheduler_type="cosine",
    seed=SEED,
    dataloader_num_workers=4,
    tf32=True,
    dataset_text_field="text",
    packing=True,
    gradient_checkpointing=True,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_text,
    eval_dataset=eval_text,
    args=sft_config,
)

train_result = trainer.train(resume_from_checkpoint=True)
print(train_result)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/37739 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/37739 [00:00<?, ? examples/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2515 > 2048). Running this sequence through the model will result in indexing errors


Packing train dataset:   0%|          | 0/37739 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/771 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/771 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss


TrainOutput(global_step=2105, training_loss=0.0, metrics={'train_runtime': 0.0069, 'train_samples_per_second': 4903233.269, 'train_steps_per_second': 306552.2, 'total_flos': 3.575039696803062e+17, 'train_loss': 0.0})


In [11]:
import os
os.makedirs(CONFIG["output_dir"], exist_ok=True)
trainer.save_model(CONFIG["output_dir"])
tokenizer.save_pretrained(CONFIG["output_dir"])
print(f"Adapter + tokenizer saved → {CONFIG['output_dir']}")


Adapter + tokenizer saved → /content/drive/MyDrive/DL/midterm/attempt2_1p7b
